# Preprocesamiento Completo del Dataset de Meningitis

En este cuadernillo se aplica todo el flujo de preprocesamiento con estilo de los cuadernillos del inge:
1. Carga y limpieza estructural
2. Separacion 80% entrenamiento y 20% prueba
3. Manejo de valores nulos
4. Codificacion de variables categoricas
5. Normalizacion de caracteristicas
6. Balanceo de clases
7. Conversor a tensores (PyTorch)

In [98]:
# ==========================================
# 1. IMPORTACION DE LIBRERIAS
# ==========================================
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from IPython.display import display

%matplotlib inline

## Paso 1: Carga del dataset
Se carga el CSV original y se revisa estructura, columnas y nulos iniciales.

In [99]:
ruta_archivo = os.path.join('Datasets Primer Parcial', '9-Meningitis Dataset', 'mening missing 12.csv')
df = pd.read_csv(ruta_archivo)

print('Dimensiones originales:', df.shape)
print('Columnas:')
print(df.columns.tolist())
print('Valores nulos por columna:')
print(df.isnull().sum())
display(df.head())

Dimensiones originales: (1200, 14)
Columnas:
['Patient_ID', 'Age', 'Gender', 'WBC_Count', 'Protein_Level', 'Glucose_Level', 'Pathogen_Present', 'Diagnosis', 'Outcome', 'Hemoglobin', 'WBC_Blood_Count', 'Platelets', 'CRP_Level', 'Risk_Level']
Valores nulos por columna:
Patient_ID           0
Age                  7
Gender               9
WBC_Count            8
Protein_Level       10
Glucose_Level        8
Pathogen_Present     8
Diagnosis           12
Outcome             10
Hemoglobin          19
WBC_Blood_Count     10
Platelets           12
CRP_Level           13
Risk_Level           0
dtype: int64


,Patient_ID,Age,Gender,WBC_Count,Protein_Level,Glucose_Level,Pathogen_Present,Diagnosis,Outcome,Hemoglobin,WBC_Blood_Count,Platelets,CRP_Level,Risk_Level
0,1,101.0,Female,8624.0,16.0,83.0,No,Viral,Recovered,15.0,7269.0,160949.0,71.0,Moderate Risk
1,2,78.0,Male,22623.0,200.0,41.0,No,Unknown,Recovered,18.0,6532.0,371741.0,41.0,High Risk
2,3,8.0,Female,12908.0,39.0,3.0,No,Unknown,Recovered,16.0,7417.0,180403.0,22.0,Moderate Risk
3,4,104.0,Female,15072.0,58.0,36.0,Yes,Bacterial,Recovered,7.0,13792.0,132254.0,48.0,Moderate Risk
4,5,38.0,Female,18623.0,152.0,34.0,Yes,Bacterial,Recovered,5.0,17054.0,134941.0,28.0,High Risk


## Paso 2: Limpieza estructural y split 80/20
Se elimina `Patient_ID` por ser identificador y luego se separa entrenamiento (80%) y prueba (20%) con indices aleatorios al estilo cuadernillo.

In [100]:
import torch

if 'Patient_ID' in df.columns:
    df = df.drop(columns=['Patient_ID'])

target_col = 'Risk_Level'

torch.manual_seed(42)

n_total = df.shape[0]
n_test = int(0.2 * n_total)
n_train = n_total - n_test

indices = torch.randperm(n_total).tolist()
train_indices = indices[:n_train]
test_indices = indices[n_train:]

train_df = df.iloc[train_indices].copy()
test_df = df.iloc[test_indices].copy()

print(f'Train: {len(train_df)}/{n_total} ({len(train_df)/n_total*100:.1f}%)')
print(f'Test : {len(test_df)}/{n_total} ({len(test_df)/n_total*100:.1f}%)')
print('Distribucion en train:')
print(train_df[target_col].value_counts())
print('Distribucion en test:')
print(test_df[target_col].value_counts())

Train: 960/1200 (80.0%)
Test : 240/1200 (20.0%)
Distribucion en train:
Risk_Level
Low Risk         449
High Risk        399
Moderate Risk    112
Name: count, dtype: int64
Distribucion en test:
Risk_Level
Low Risk         109
High Risk        106
Moderate Risk     25
Name: count, dtype: int64


## Paso 3: Separar X e y
Se separan variables predictoras y variable objetivo para train y test.

In [101]:
feature_cols = [c for c in df.columns if c != target_col]

X_train_raw = train_df[feature_cols].copy()
X_test_raw = test_df[feature_cols].copy()
y_train_raw = train_df[target_col].copy()
y_test_raw = test_df[target_col].copy()

print('Features:', feature_cols)
print(X_train_raw.dtypes)

Features: ['Age', 'Gender', 'WBC_Count', 'Protein_Level', 'Glucose_Level', 'Pathogen_Present', 'Diagnosis', 'Outcome', 'Hemoglobin', 'WBC_Blood_Count', 'Platelets', 'CRP_Level']
Age                 float64
Gender                  str
WBC_Count           float64
Protein_Level       float64
Glucose_Level       float64
Pathogen_Present        str
Diagnosis               str
Outcome                 str
Hemoglobin          float64
WBC_Blood_Count     float64
Platelets           float64
CRP_Level           float64
dtype: object


## Paso 4: Manejo de valores nulos
Como en los cuadernillos: mediana para numericas y moda para categoricas, calculado en train y aplicado en train/test.

In [102]:
numeric_cols = X_train_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train_raw.select_dtypes(include=['object', 'string', 'category']).columns.tolist()

print('Nulos antes (train):', X_train_raw.isnull().sum().sum())
print('Nulos antes (test) :', X_test_raw.isnull().sum().sum())

for col in numeric_cols:
    med = X_train_raw[col].median()
    X_train_raw[col] = X_train_raw[col].fillna(med)
    X_test_raw[col] = X_test_raw[col].fillna(med)

for col in categorical_cols:
    moda = X_train_raw[col].mode()[0]
    X_train_raw[col] = X_train_raw[col].fillna(moda)
    X_test_raw[col] = X_test_raw[col].fillna(moda)

print('Nulos despues (train):', X_train_raw.isnull().sum().sum())
print('Nulos despues (test) :', X_test_raw.isnull().sum().sum())

Nulos antes (train): 99
Nulos antes (test) : 27
Nulos despues (train): 0
Nulos despues (test) : 0


## Paso 5: Funciones 
Se define la funcion `featureNormalize(X)` exactamente en el estilo de los cuadernillos del inge. También se incluye sigmoide como funcion util para clasificacion posterior.

In [103]:
def featureNormalize(X):
    X_norm = X.copy()
    mu = np.zeros(X.shape[1])
    sigma = np.zeros(X.shape[1])

    mu = np.mean(X, axis = 0)
    sigma = np.std(X, axis = 0)
    X_norm = (X - mu) / sigma

    return X_norm, mu, sigma

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

## Paso 6: Codificacion de categoricas y normalizacion de numericas
Se codifica y con LabelEncoder, se codifican categoricas de entrada con OneHotEncoder y se normalizan numericas con `featureNormalize` (train) aplicando `mu` y `sigma` a test.

In [104]:
target_encoder = LabelEncoder()
y_train = target_encoder.fit_transform(y_train_raw)
y_test = target_encoder.transform(y_test_raw)

if categorical_cols:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_train_cat = ohe.fit_transform(X_train_raw[categorical_cols])
    X_test_cat = ohe.transform(X_test_raw[categorical_cols])
    cat_feature_names = ohe.get_feature_names_out(categorical_cols)
else:
    X_train_cat = np.empty((len(X_train_raw), 0))
    X_test_cat = np.empty((len(X_test_raw), 0))
    cat_feature_names = np.array([])

if numeric_cols:
    X_train_num, mu_norm, sigma_norm = featureNormalize(X_train_raw[numeric_cols].to_numpy())
    X_test_num = (X_test_raw[numeric_cols].to_numpy() - mu_norm) / sigma_norm
else:
    X_train_num = np.empty((len(X_train_raw), 0))
    X_test_num = np.empty((len(X_test_raw), 0))
    mu_norm = np.array([])
    sigma_norm = np.array([])

num_feature_names = np.array(numeric_cols)
final_feature_names = list(num_feature_names) + list(cat_feature_names)

X_train_final = np.concatenate([X_train_num, X_train_cat], axis=1)
X_test_final = np.concatenate([X_test_num, X_test_cat], axis=1)

X_train_final_df = pd.DataFrame(X_train_final, columns=final_feature_names, index=X_train_raw.index)
X_test_final_df = pd.DataFrame(X_test_final, columns=final_feature_names, index=X_test_raw.index)

print('X_train final:', X_train_final_df.shape)
print('X_test final :', X_test_final_df.shape)
display(X_train_final_df.head())

X_train final: (960, 17)
X_test final : (240, 17)


,Age,WBC_Count,Protein_Level,Glucose_Level,Hemoglobin,WBC_Blood_Count,Platelets,CRP_Level,Gender_Female,Gender_Male,Pathogen_Present_No,Pathogen_Present_Yes,Diagnosis_Bacterial,Diagnosis_Unknown,Diagnosis_Viral,Outcome_Deceased,Outcome_Recovered
342,-0.075952,-0.443963,-0.896425,0.371395,0.713636,-1.115648,-0.294727,-0.978229,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
184,-0.560699,0.900904,1.100544,-1.183692,0.895357,-0.438397,-0.512186,-0.103339,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
916,2.509363,-1.563384,1.002791,1.983030,0.350193,-1.410819,1.963597,0.581359,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
169,1.297497,-0.674558,1.407770,-1.070595,0.350193,-0.698317,1.805413,-0.902152,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
710,-0.681885,2.268315,1.994293,-0.448560,-1.648744,0.675634,-0.776718,1.988791,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0


## Paso 7: Balanceo de clases en entrenamiento
Se realiza undersampling solo en entrenamiento para evitar fuga de informacion hacia prueba.

In [105]:
train_processed = X_train_final_df.copy()
train_processed[target_col] = y_train
test_processed = X_test_final_df.copy()
test_processed[target_col] = y_test

print('Distribucion antes del balanceo (train):')
print(train_processed[target_col].value_counts().sort_index())

min_count = train_processed[target_col].value_counts().min()
train_balanced = train_processed.groupby(target_col, group_keys=False).sample(n=min_count, random_state=42)

print('Distribucion despues del balanceo (train):')
print(train_balanced[target_col].value_counts().sort_index())
print('Train balanceado:', train_balanced.shape)
print('Test sin balancear:', test_processed.shape)

Distribucion antes del balanceo (train):
Risk_Level
0    399
1    449
2    112
Name: count, dtype: int64
Distribucion despues del balanceo (train):
Risk_Level
0    112
1    112
2    112
Name: count, dtype: int64
Train balanceado: (336, 18)
Test sin balancear: (240, 18)


In [113]:
print('Train antes del balanceo:')
porc_train = train_processed['Risk_Level'].value_counts(normalize=True).sort_index()
print((round(porc_train * 100, 2)).astype(str) + ' %')

print('\nTrain despues del balanceo:')
porc_bal = train_balanced['Risk_Level'].value_counts(normalize=True).sort_index()
print((round(porc_bal * 100, 2)).astype(str) + ' %')

Train antes del balanceo:
Risk_Level
0    41.56 %
1    46.77 %
2    11.67 %
Name: proportion, dtype: str

Train despues del balanceo:
Risk_Level
0    33.33 %
1    33.33 %
2    33.33 %
Name: proportion, dtype: str


## Paso 7: Conversor a tensores (PyTorch)
En este paso se convierten los conjuntos preprocesados a tensores para usarlos directamente en modelos de PyTorch.

In [106]:
import torch

X_train_np = train_balanced.drop(columns=[target_col]).to_numpy().astype(np.float32)
y_train_np = train_balanced[target_col].to_numpy().astype(np.int64)
X_test_np = test_processed.drop(columns=[target_col]).to_numpy().astype(np.float32)
y_test_np = test_processed[target_col].to_numpy().astype(np.int64)

X_train = torch.from_numpy(X_train_np)
y_train = torch.from_numpy(y_train_np)
X_test = torch.from_numpy(X_test_np)
y_test = torch.from_numpy(y_test_np)

print('Preparación terminada:')
print(f'X_train shape: {X_train.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_test shape: {y_test.shape}')

Preparación terminada:
X_train shape: torch.Size([336, 17])
y_train shape: torch.Size([336])
X_test shape: torch.Size([240, 17])
y_test shape: torch.Size([240])
